# Final Dataset Analysis

This notebook performs high-level metadata analysis on MosaicMRI H5 files (split counts, header-derived stats, anatomy summaries).

Expected inputs:
- `MosaicMRI/multicoil_train`
- `MosaicMRI/multicoil_val`
- `MosaicMRI/multicoil_test`

Notes:
- Paths are intentionally generic for release.
- Cell outputs are cleared for a clean commit.


In [ ]:
DATASET_FOLDERS = [
    #'MosaicMRI/multicoil_test',
    #'MosaicMRI/multicoil_train',
   'MosaicMRI/multicoil_val',
]

from pathlib import Path
import random

# Resolve relative folders from *this notebook's* directory
NB_DIR = Path.cwd()  # assume notebook kernel cwd is the notebook folder; if not, fallback below
try:
    NB_DIR = Path(__file__).parent  # type: ignore[name-defined]
except Exception:
    pass

RESOLVED_DATASET_FOLDERS = [str((NB_DIR / Path(p)).resolve()) for p in DATASET_FOLDERS]

folder = random.choice(RESOLVED_DATASET_FOLDERS)
files = [f for f in Path(folder).iterdir() if f.name.endswith('.h5')]
if not files:
    raise FileNotFoundError(f"No .h5 files found in: {folder}")

random_file = str(random.choice(files))

#random_file = "meas_MID00415_FID121697_COR_T1.h5"
#random_file = folder + "/" + random_file

print(random_file)


In [ ]:
# Pretty-print ISMRMRD header and show dataset shapes (robust formatting, plain text output)
import xml.dom.minidom as minidom
import json
import h5py
import numpy as np
import xml.etree.ElementTree as ET


def _strip_xml_namespaces(xml_str: str) -> str:
    try:
        root = ET.fromstring(xml_str)
    except Exception:
        return xml_str
    for elem in root.iter():
        # Strip namespace from tag
        if isinstance(elem.tag, str) and '}' in elem.tag:
            elem.tag = elem.tag.split('}', 1)[1]
        # Strip namespace from attribute names
        if elem.attrib:
            new_attrib = {}
            for k, v in elem.attrib.items():
                if '}' in k:
                    k = k.split('}', 1)[1]
                new_attrib[k] = v
            elem.attrib.clear()
            elem.attrib.update(new_attrib)
    return ET.tostring(root, encoding='utf-8').decode('utf-8')


def _pretty_xml(s: str) -> str | None:
    try:
        # Remove namespaces to avoid ns0 prefixes
        no_ns = _strip_xml_namespaces(s)
        root = ET.fromstring(no_ns)
        rough = ET.tostring(root, encoding="utf-8")
        pretty = minidom.parseString(rough).toprettyxml(indent="  ")
        pretty = "\n".join([line for line in pretty.splitlines() if line.strip()])
        return pretty
    except Exception:
        return None


def _display_block(title: str, body: str, lang: str = "xml") -> None:
    # Plain text output, no markdown
    print(title)
    print(body)


def _to_str_from_dataset(ds: h5py.Dataset) -> str:
    # Try asstr (available in h5py>=3) for string datasets
    try:
        return ds.asstr()[()].strip()
    except Exception:
        pass
    # Raw read
    obj = ds[()]
    # If bytes-like directly
    if isinstance(obj, (bytes, bytearray)):
        return bytes(obj).replace(b"\x00", b"").decode("utf-8", errors="replace").strip()
    # If numpy scalar string/bytes
    if np.isscalar(obj):
        if isinstance(obj.item(), (bytes, bytearray)):
            return obj.item().replace(b"\x00", b"").decode("utf-8", errors="replace").strip()
        try:
            return str(obj.item())
        except Exception:
            pass
    # If numpy array of uint8/bytes
    if isinstance(obj, np.ndarray):
        if obj.dtype == np.uint8:
            return bytes(obj.tobytes()).replace(b"\x00", b"").decode("utf-8", errors="replace").strip()
        if obj.dtype.kind == 'S':  # fixed-width bytes strings
            try:
                joined = b"".join(obj.tolist())
            except Exception:
                joined = bytes(obj.tobytes())
            return joined.replace(b"\x00", b"").decode("utf-8", errors="replace").strip()
        if obj.dtype.kind == 'U':  # unicode strings
            try:
                return "".join(obj.tolist()).strip()
            except Exception:
                return str(obj)
    # Fallback
    return str(obj)


def _extract_xml_text(root: ET.Element, path: str) -> str | None:
    el = root.find(path)
    if el is None or el.text is None:
        return None
    txt = el.text.strip()
    return txt or None


def _find_user_parameter_string(root: ET.Element, param_name: str) -> str | None:
    # Schema pattern observed in Siemens exports:
    # <userParameters>
    # </userParameters>
    for ups in root.findall('.//userParameters/userParameterString'):
        name_el = ups.find('name')
        if name_el is None or name_el.text is None:
            continue
        if name_el.text.strip() != param_name:
            continue
        val_el = ups.find('value')
        if val_el is None or val_el.text is None:
            return None
        v = val_el.text.strip()
        return v or None
    return None


with h5py.File(random_file, 'r') as f:
    # print keys
    print(list(f.keys()))

    # Decode header robustly to Python str
    header_str = _to_str_from_dataset(f["ismrmrd_header"]) or ""

    # Print a couple of commonly useful text fields from the XML header
    # Print anatomy/contrast attrs if present
    try:
        def _decode_attr(val):
            if isinstance(val, (bytes, bytearray)):
                return val.decode('utf-8', errors='replace')
            try:
                if hasattr(val, 'shape') and val.shape == ():
                    val = val[()]
                    if isinstance(val, (bytes, bytearray)):
                        return val.decode('utf-8', errors='replace')
            except Exception:
                pass
            return str(val)

        attrs = dict(f.attrs)
        def _get_attr(keys):
            for k in keys:
                if k in attrs:
                    return _decode_attr(attrs[k])
            return None
        anatomy_attr = _get_attr(['anatomy','ANATOMY','body_part','BodyPartExamined','bodyPartExamined'])
        contrast_attr = _get_attr(['contrast','CONTRAST','sequence','Sequence'])
        print('anatomy attr:', anatomy_attr)
        print('contrast attr:', contrast_attr)
    except Exception:
        pass

    #Try pretty XML first
    printed = False
    pretty_xml = _pretty_xml(header_str)
    if pretty_xml:
        _display_block("ISMRMRD Header (XML)", pretty_xml, lang="xml")
        printed = True

    # If not XML or parsing failed, try JSON
    if not printed:
        try:
            obj = json.loads(header_str)
            _display_block("ISMRMRD Header (JSON)", json.dumps(obj, indent=2), lang="json")
            printed = True
        except Exception:
            pass

    # Fallback: raw text block
    if not printed:
        _display_block("ISMRMRD Header (raw)", header_str, lang="text")

    #Print dataset shapes
    print("kspace shape:", f["kspace"].shape)
    print("reconstruction_rss shape:", f["reconstruction_rss"].shape)
    print("attrs:", dict(f.attrs))


In [ ]:
# Dataset-wide statistics across all files and splits (plain text output)
import os
import json
import math
import h5py
import numpy as np
from collections import Counter, defaultdict
from statistics import mean
import xml.etree.ElementTree as ET

def _safe_mean(vals):
    vals = [v for v in vals if v is not None]
    return float(mean(vals)) if vals else float('nan')

def _safe_median(vals):
    vals = [v for v in vals if v is not None]
    return float(np.median(vals)) if vals else float('nan')

def _safe_min(vals):
    vals = [v for v in vals if v is not None]
    return min(vals) if vals else None

def _safe_max(vals):
    vals = [v for v in vals if v is not None]
    return max(vals) if vals else None

def _extract_text(elem):
    return elem.text.strip() if elem is not None and elem.text is not None else None

def _strip_xml_ns_text(xml_str: str) -> ET.Element | None:
    try:
        root = ET.fromstring(xml_str)
    except Exception:
        return None
    for e in root.iter():
        if isinstance(e.tag, str) and '}' in e.tag:
            e.tag = e.tag.split('}', 1)[1]
        if e.attrib:
            e.attrib = {(k.split('}', 1)[1] if '}' in k else k): v for k, v in e.attrib.items()}
    return root

def _as_float(x: str | None) -> float | None:
    if x is None:
        return None
    try:
        return float(x)
    except Exception:
        try:
            return float(str(x))
        except Exception:
            return None

def _get_voxel_size_xy_mm(root: ET.Element) -> tuple[float | None, float | None]:
    # ISMRMRD convention: encodedSpace / reconSpace fieldOfView_mm and matrixSize -> voxel = fov / matrix
    # We prefer reconSpace if present, else fall back to encodedSpace.
    def _fov_mm(path_prefix: str) -> tuple[float | None, float | None]:
        fx = _as_float(_extract_text(root.find(path_prefix + '/fieldOfView_mm/x')))
        fy = _as_float(_extract_text(root.find(path_prefix + '/fieldOfView_mm/y')))
        return fx, fy

    def _mat_xy(path_prefix: str) -> tuple[int | None, int | None]:
        mx_txt = _extract_text(root.find(path_prefix + '/matrixSize/x'))
        my_txt = _extract_text(root.find(path_prefix + '/matrixSize/y'))
        mx = None
        my = None
        if mx_txt is not None:
            try:
                mx = int(mx_txt)
            except Exception:
                try:
                    mx = int(float(mx_txt))
                except Exception:
                    mx = None
        if my_txt is not None:
            try:
                my = int(my_txt)
            except Exception:
                try:
                    my = int(float(my_txt))
                except Exception:
                    my = None
        return mx, my

    for space in ['.//encoding/reconSpace', './/encoding/encodedSpace']:
        fx, fy = _fov_mm(space)
        mx, my = _mat_xy(space)
        if fx is None or fy is None or mx in (None, 0) or my in (None, 0):
            continue
        return (fx / mx, fy / my)
    return (None, None)

def parse_header_fields(header_str: str):
    root = _strip_xml_ns_text(header_str)
    if root is None:
        return {
            'patient_id': None,
            'field_strength_T': None,
            'receiver_channels': None,
            'encoded_matrix': None,
            'recon_matrix': None,
            'inplane_res_mm': (None, None),
            'patient_gender': None,
            'patient_weight_kg': None,
        }
    # patient id (prefer subjectInformation/patientID, fallback to userParameters PatientLOID)
    patient_id = _extract_text(root.find('.//subjectInformation/patientID'))
    if not patient_id:
        for ups in root.findall('.//userParameters/userParameterString'):
            name = _extract_text(ups.find('name'))
            if name == 'PatientLOID':
                patient_id = _extract_text(ups.find('value'))
                if patient_id:
                    break
    # field strength
    fs_txt = _extract_text(root.find('.//acquisitionSystemInformation/systemFieldStrength_T'))
    field_strength_T = None
    if fs_txt:
        try:
            field_strength_T = float(fs_txt)
        except Exception:
            pass
    # receiver channels
    rc_txt = _extract_text(root.find('.//acquisitionSystemInformation/receiverChannels'))
    receiver_channels = None
    if rc_txt:
        try:
            receiver_channels = int(rc_txt)
        except Exception:
            try:
                receiver_channels = int(float(rc_txt))
            except Exception:
                pass
    # patient gender and weight
    patient_gender = _extract_text(root.find('.//subjectInformation/patientGender'))
    patient_weight_kg = _as_float(_extract_text(root.find('.//subjectInformation/patientWeight_kg')))
    # matrix sizes
    def _matrix(node_path):
        n = root.find(node_path)
        if n is None:
            return None
        x = _extract_text(n.find('x'))
        y = _extract_text(n.find('y'))
        z = _extract_text(n.find('z'))
        try:
            return (int(x), int(y), int(z))
        except Exception:
            try:
                return (int(float(x)), int(float(y)), int(float(z)))
            except Exception:
                return None
    encoded_matrix = _matrix('.//encoding/encodedSpace/matrixSize')
    recon_matrix = _matrix('.//encoding/reconSpace/matrixSize')
    inplane_res_mm = _get_voxel_size_xy_mm(root)
    return {
        'patient_id': patient_id,
        'field_strength_T': field_strength_T,
        'receiver_channels': receiver_channels,
        'encoded_matrix': encoded_matrix,
        'recon_matrix': recon_matrix,
        'inplane_res_mm': inplane_res_mm,
        'patient_gender': patient_gender,
        'patient_weight_kg': patient_weight_kg,
    }

def infer_split(path_str: str) -> str:
    p = path_str.lower()
    if 'train' in p:
        return 'train'
    if 'val' in p or 'valid' in p:
        return 'val'
    if 'test' in p:
        return 'test'
    return 'unknown'

def read_header_str_from_file(f: h5py.File) -> str:
    # Reuse helper from previous cell if available
    try:
        return _to_str_from_dataset(f['ismrmrd_header'])
    except NameError:
        pass
    # Fallback minimal decoder
    obj = f['ismrmrd_header'][()]
    if isinstance(obj, (bytes, bytearray)):
        return bytes(obj).replace(b'\x00', b'').decode('utf-8', errors='replace').strip()
    try:
        return str(obj)
    except Exception:
        return ''



def _decode_attr(val):
    if isinstance(val, (bytes, bytearray)):
        return val.decode('utf-8', errors='replace')
    try:
        if hasattr(val, 'shape') and val.shape == ():
            val = val[()]
            if isinstance(val, (bytes, bytearray)):
                return val.decode('utf-8', errors='replace')
    except Exception:
        pass
    return str(val)

def _get_attr_from_file(f: h5py.File, keys):
    for key in keys:
        if key in f.attrs:
            return _decode_attr(f.attrs[key])
    return None



def infer_anatomy_from_attrs(f: h5py.File) -> str:
    for key in ('anatomy', 'ANATOMY', 'body_part', 'BodyPartExamined', 'bodyPartExamined'):
        if key in f.attrs:
            return _decode_attr(f.attrs[key]).upper()
    return 'UNKNOWN'

def infer_anatomy_from_attrs_path(path: str) -> str:
    try:
        with h5py.File(path, 'r') as f:
            return infer_anatomy_from_attrs(f)
    except Exception:
        return 'UNKNOWN'


all_files = []
for root_dir in DATASET_FOLDERS:
    if not os.path.isdir(root_dir):
        continue
    for name in os.listdir(root_dir):
        if name.endswith('.h5'):
            all_files.append(os.path.join(root_dir, name))

num_files = len(all_files)
if num_files == 0:
    print('No .h5 files found in DATASET_FOLDERS')
else:
    patients = set()
    field_strengths = []
    field_strength_counter = Counter()
    coils_list = []  # per file
    coils_counter = Counter()
    slices_per_file = []
    slices_by_split = defaultdict(list)
    coils_by_split = defaultdict(list)
    matrices_kspace = Counter()  # (H, W) from kspace
    matrices_encoded = Counter()  # (x,y,z)
    matrices_recon = Counter()  # (x,y,z)

    # anatomy aggregates for the pie chart
    by_anatomy = defaultdict(lambda: {"n_volumes": 0, "sum_n_slices": 0})
    unknown_examples = []  # (fname, attrs) debug samples

    # reconSpace x/y stats across files (use metadata recon_matrix x,y when present)
    recon_x_vals = []
    recon_y_vals = []

    # in-plane resolution per volume (mm) from header (FOV / matrix)
    inplane_x_mm = []
    inplane_y_mm = []
    inplane_area_mm2 = []
    inplane_by_split = defaultdict(lambda: {'x': [], 'y': [], 'area': []})

    # Patient gender and weight statistics
    gender_counter = Counter()
    weight_vals = []
    weight_by_gender = defaultdict(list)

    for fp in all_files:
        split = infer_split(fp)
        try:
            with h5py.File(fp, 'r') as f:
                # header
                header_str = read_header_str_from_file(f) or ''
                meta = parse_header_fields(header_str)
                anat = infer_anatomy_from_attrs(f)
                if anat == 'UNKNOWN' and len(unknown_examples) < 10:
                    unknown_examples.append((os.path.basename(fp), dict(f.attrs)))

                if meta.get('patient_id'):
                    patients.add(meta['patient_id'])
                # field strength
                fs = meta.get('field_strength_T')
                if fs is not None:
                    field_strengths.append(fs)
                    field_strength_counter[round(fs, 3)] += 1
                # coils: prefer metadata else kspace dim
                rc = meta.get('receiver_channels')
                # shapes
                slices = 0
                if 'kspace' in f:
                    ks_shape = f['kspace'].shape
                    # Expect (slices, coils, H, W) for multicoil
                    if len(ks_shape) >= 4:
                        slices = int(ks_shape[0])
                        coils_dim = int(ks_shape[1])
                        H, W = int(ks_shape[-2]), int(ks_shape[-1])
                        matrices_kspace[(H, W)] += 1
                    else:
                        slices = int(ks_shape[0]) if len(ks_shape) > 0 else 0
                        coils_dim = int(ks_shape[1]) if len(ks_shape) > 1 else None
                    slices_per_file.append(slices)
                    slices_by_split[split].append(slices)
                    if rc is None and coils_dim is not None:
                        rc = coils_dim
                # anatomy aggregates
                by_anatomy[anat]["n_volumes"] += 1
                by_anatomy[anat]["sum_n_slices"] += int(slices)

                if rc is not None:
                    coils_list.append(rc)
                    coils_by_split[split].append(rc)
                    coils_counter[int(rc)] += 1
                # matrices from metadata
                if meta.get('encoded_matrix'):
                    matrices_encoded[meta['encoded_matrix']] += 1
                if meta.get('recon_matrix'):
                    matrices_recon[meta['recon_matrix']] += 1
                    x, y, _z = meta['recon_matrix']
                    recon_x_vals.append(x)
                    recon_y_vals.append(y)

                # in-plane resolution
                rx, ry = meta.get('inplane_res_mm', (None, None))
                if rx is not None and ry is not None:
                    inplane_x_mm.append(rx)
                    inplane_y_mm.append(ry)
                    inplane_area_mm2.append(rx * ry)
                    inplane_by_split[split]['x'].append(rx)
                    inplane_by_split[split]['y'].append(ry)
                    inplane_by_split[split]['area'].append(rx * ry)

                # gender and weight
                gender = meta.get('patient_gender')
                weight = meta.get('patient_weight_kg')
                if gender:
                    gender_counter[gender] += 1
                    if weight is not None:
                        weight_by_gender[gender].append(weight)
                if weight is not None:
                    weight_vals.append(weight)
        except Exception:
            continue

    # Compute aggregates
    unique_patients = len(patients)
    total_slices = sum(slices_per_file) if slices_per_file else 0
    avg_slices = _safe_mean(slices_per_file)
    min_slices = min(slices_per_file) if slices_per_file else 0
    max_slices = max(slices_per_file) if slices_per_file else 0
    min_coils = min(coils_list) if coils_list else None
    max_coils = max(coils_list) if coils_list else None

    # For export: a plain dict
    by_anatomy = {k: dict(v) for k, v in by_anatomy.items() if v["n_volumes"] > 0}

    print('Dataset summary')
    print(f'- Files: {num_files}')
    print(f'- Unique patients: {unique_patients}')

    print('\nAnatomy summary (from H5 attrs)')
    for k, v in sorted(by_anatomy.items(), key=lambda kv: kv[1]['n_volumes'], reverse=True):
        print(f"- {k}: {v['n_volumes']} volumes, {v['sum_n_slices']} slices")

    if unknown_examples:
        print('\nExamples that mapped to UNKNOWN (filename, attrs):')
        for fname, desc in unknown_examples:
            print(f"- {fname}: {desc}")

    print('\nField strength (Tesla) distribution')
    if field_strength_counter:
        for val, cnt in sorted(field_strength_counter.items(), key=lambda x: x[0]):
            print(f'- {val}: {cnt}')
    else:
        print('- (none)')

    print('\nSlices per file (overall)')
    print(f'- Total slices: {total_slices}')
    print(f'- Avg slices: {avg_slices:.2f}')
    print(f'- Min slices: {min_slices}')
    print(f'- Max slices: {max_slices}')

    for sp in ['train', 'val', 'test', 'unknown']:
        svals = slices_by_split.get(sp, [])
        if not svals:
            continue
        print(f'- {sp}: n={len(svals)}, total={sum(svals)}, avg={_safe_mean(svals):.2f}, min={min(svals)}, max={max(svals)}')

    print('\nReceiver channels (per scan)')
    if coils_counter:
        print(f'- Min coils: {min_coils}')
        print(f'- Max coils: {max_coils}')
        for k, v in sorted(coils_counter.items()):
            print(f'- {k}: {v}')
    else:
        print('- (none)')

    print('\nMatrix sizes')
    if matrices_kspace:
        for (H, W), v in sorted(matrices_kspace.items()):
            print(f'- kspace HxW {H}x{W}: {v}')
    else:
        print('- kspace: (none)')
    if matrices_encoded:
        for (x, y, z), v in sorted(matrices_encoded.items()):
            print(f'- encodedSpace {x}x{y}x{z}: {v}')
    if matrices_recon:
        for (x, y, z), v in sorted(matrices_recon.items()):
            print(f'- reconSpace {x}x{y}x{z}: {v}')

    print('\nReconSpace x/y summary (from encoding/reconSpace/matrixSize)')
    if recon_x_vals and recon_y_vals:
        print(f'- reconSpace x: min={min(recon_x_vals)}, median={_safe_median(recon_x_vals):.1f}, max={max(recon_x_vals)}')
        print(f'- reconSpace y: min={min(recon_y_vals)}, median={_safe_median(recon_y_vals):.1f}, max={max(recon_y_vals)}')
    else:
        print('- (none found in headers)')

    print('\nIn-plane resolution per volume (mm) (computed as FOV_mm / matrixSize)')
    if inplane_x_mm and inplane_y_mm:
        print(f'- x (mm): min={_safe_min(inplane_x_mm):.4f}, median={_safe_median(inplane_x_mm):.4f}, max={_safe_max(inplane_x_mm):.4f}')
        print(f'- y (mm): min={_safe_min(inplane_y_mm):.4f}, median={_safe_median(inplane_y_mm):.4f}, max={_safe_max(inplane_y_mm):.4f}')
        print(f'- area (mm^2): min={_safe_min(inplane_area_mm2):.4f}, median={_safe_median(inplane_area_mm2):.4f}, max={_safe_max(inplane_area_mm2):.4f}')
        for sp in ['train', 'val', 'test', 'unknown']:
            vx = inplane_by_split.get(sp, {}).get('x', [])
            vy = inplane_by_split.get(sp, {}).get('y', [])
            if not vx or not vy:
                continue
            print(f'- {sp}: x min/med/max={_safe_min(vx):.4f}/{_safe_median(vx):.4f}/{_safe_max(vx):.4f}, y min/med/max={_safe_min(vy):.4f}/{_safe_median(vy):.4f}/{_safe_max(vy):.4f}')
    else:
        print('- (none found in headers; expected encoding/*Space/fieldOfView_mm and matrixSize)')

    # Patient gender and weight statistics
    print('\nPatient gender count:')
    n_male = gender_counter.get('M', 0)
    n_female = gender_counter.get('F', 0)
    print(f'- Males: {n_male}')
    print(f'- Females: {n_female}')

    print('\nPatient weight (kg) statistics:')
    if weight_vals:
        print(f'- min: {min(weight_vals):.2f}, median: {np.median(weight_vals):.2f}, max: {max(weight_vals):.2f}, mean: {np.mean(weight_vals):.2f}')
        for g in ['M', 'F']:
            vals = weight_by_gender.get(g, [])
            if vals:
                label = 'Males' if g == 'M' else 'Females'
                print(f'- {label}: min={min(vals):.2f}, median={np.median(vals):.2f}, max={max(vals):.2f}, mean={np.mean(vals):.2f}')
    else:
        print('- (none found)')

# --- Sequence distribution (SpinEcho / GradientEcho / bSSFP) ---
from collections import Counter

def infer_sequence_from_attrs(f: h5py.File):
    seq_type = _get_attr_from_file(f, ('sequence_type', 'sequenceType', 'sequence'))
    proto = _get_attr_from_file(f, ('contrast', 'CONTRAST', 'sequence', 'Sequence'))
    seqfile = _get_attr_from_file(f, ('tSequenceFileName', 'sequenceFileName'))

    s = ('{} {} {}').format(seq_type or '', proto or '', seqfile or '').upper()
    if any(k in s for k in ['BSSFP', 'TRUEFISP', 'FISP']):
        return 'bSSFP'
    if any(k in s for k in ['GRE', 'GRADIENT', 'FLASH', 'SPGR', 'TFL', 'FFE']):
        return 'GradientEcho'
    if any(k in s for k in ['TURBOSPINECHO', 'TURBO SPIN ECHO', 'TSE', 'FSE', 'SE', 'SPINECHO', 'SPIN ECHO']):
        return 'SpinEcho'
    return None

seq_counts = Counter()
for fp in all_files:
    with h5py.File(fp, 'r') as f:
        seq = infer_sequence_from_attrs(f)
    if seq:
        seq_counts[seq] += 1

print('Sequence distribution:')
for k, v in seq_counts.most_common():
    print('  {}: {}'.format(k, v))


In [ ]:
# Print average FOV and average resolution per anatomy
from collections import Counter


anatomy_fov = {}
anatomy_res = {}
anatomy_coils = {}
anatomy_coils_hist = {}

for anat, v in by_anatomy.items():
    # Find all files for this anatomy
    anat_files = [fp for fp in all_files if infer_anatomy_from_attrs_path(fp) == anat]
    fov_xs, fov_ys, res_xs, res_ys, coils = [], [], [], [], []
    for fp in anat_files:
        try:
            with h5py.File(fp, 'r') as f:
                header_str = read_header_str_from_file(f) or ''
                root = _strip_xml_ns_text(header_str)
                if root is None:
                    continue
                # FOV (prefer reconSpace, fallback to encodedSpace)
                fx, fy = None, None
                for space in ['.//encoding/reconSpace', './/encoding/encodedSpace']:
                    x = _extract_text(root.find(space + '/fieldOfView_mm/x'))
                    y = _extract_text(root.find(space + '/fieldOfView_mm/y'))
                    if x and y:
                        try:
                            fx, fy = float(x), float(y)
                        except Exception:
                            continue
                        break
                # Resolution (voxel size, mm)
                rx, ry = None, None
                for space in ['.//encoding/reconSpace', './/encoding/encodedSpace']:
                    mx = _extract_text(root.find(space + '/matrixSize/x'))
                    my = _extract_text(root.find(space + '/matrixSize/y'))
                    if mx and my and fx and fy:
                        try:
                            rx, ry = fx / float(mx), fy / float(my)
                        except Exception:
                            continue
                        break
                if fx is not None:
                    fov_xs.append(fx)
                if fy is not None:
                    fov_ys.append(fy)
                if rx is not None:
                    res_xs.append(rx)
                if ry is not None:
                    res_ys.append(ry)
                # Coils
                rc = None
                rc_txt = _extract_text(root.find('.//acquisitionSystemInformation/receiverChannels'))
                if rc_txt:
                    try:
                        rc = int(rc_txt)
                    except Exception:
                        try:
                            rc = int(float(rc_txt))
                        except Exception:
                            pass
                if rc is None and 'kspace' in f:
                    ks_shape = f['kspace'].shape
                    if len(ks_shape) >= 2:
                        rc = int(ks_shape[1])
                if rc is not None:
                    coils.append(rc)
        except Exception:
            continue
    if fov_xs and fov_ys:
        avg_fov_x = sum(fov_xs) / len(fov_xs)
        avg_fov_y = sum(fov_ys) / len(fov_ys)
        anatomy_fov[anat] = (avg_fov_x, avg_fov_y)
    if res_xs and res_ys:
        avg_res_x = sum(res_xs) / len(res_xs)
        avg_res_y = sum(res_ys) / len(res_ys)
        anatomy_res[anat] = (avg_res_x, avg_res_y)
    if coils:
        anatomy_coils[anat] = sum(coils) / len(coils)
        anatomy_coils_hist[anat] = Counter(coils)

# Attach stats into by_anatomy so the JSON export can reuse it.
for anat in by_anatomy:
    fov = anatomy_fov.get(anat)
    res = anatomy_res.get(anat)
    hist = anatomy_coils_hist.get(anat)
    avg_c = anatomy_coils.get(anat)
    if fov:
        by_anatomy[anat]['avg_fov_mm'] = [float(fov[0]), float(fov[1])]
    if res:
        by_anatomy[anat]['avg_res_mm'] = [float(res[0]), float(res[1])]
    if avg_c is not None:
        by_anatomy[anat]['avg_coils'] = float(avg_c)
    if hist:
        by_anatomy[anat]['coils_hist'] = {str(k): int(v) for k, v in hist.items()}
        # mode: highest count, tie -> smaller coil count
        mode_k = max(hist.items(), key=lambda kv: (kv[1], -kv[0]))[0]
        by_anatomy[anat]['coils_mode'] = int(mode_k)

for anat in by_anatomy:
    fov = anatomy_fov.get(anat)
    res = anatomy_res.get(anat)
    avg_coils = anatomy_coils.get(anat, None)
    if fov:
        fov_s = f"x={fov[0]:.2f}mm, y={fov[1]:.2f}mm"
    else:
        fov_s = "N/A"
    if res:
        res_s = f"x={res[0]:.4f}mm, y={res[1]:.4f}mm"
    else:
        res_s = "N/A"
    coils_s = f"{avg_coils:.2f}" if avg_coils is not None else "N/A"
    print(f"- {anat}: avg FOV {fov_s} | avg res {res_s} | avg coils={coils_s}")
